# Base Year Data Engineering
**Purpose:** Build the parcel-level attributed dataset and TAZ-level input files
for the 2026 Housing EIS Travel Demand Model base year.

**Stages:**
1. Data Acquisition (web services)
2. Parcel Classification (VHR flag, TAU type, unit changes, spatial joins)
3. Occupancy Rate Application
4. IDW Interpolation â€” **requires ArcPy / Spatial Analyst**
5. Census / Socioeconomic Attribution
6. Campground Processing
7. School Enrollment Processing
8. TAZ Aggregation & Output CSV Export
9. QA & Basin Summary

**Checkpoints** are saved as Parquet (no pickle version lock-in).

> Cells tagged `[ARCPY]` require ArcGIS Pro. All others run in any pandas environment.

## Configuration
All constants live here. Update this cell when running for a new base year.

In [ ]:
from pathlib import Path

# Year filters
PARCEL_YEAR          = 2022
SCHOOL_YEAR          = "2021-2022"
CENSUS_YEAR          = 2022
# Occupancy rate timeframes to average (ISO date strings from the feature service)
OCCUPANCY_TIMEFRAMES = ["2022-06-01", "2022-08-01", "2022-09-01"]

# Fallback occupancy rates (used only when IDW also fails) 
DEFAULT_TAU_OCCUPANCY = 0.592337
DEFAULT_VHR_OCCUPANCY = 0.422337

# Population adjustment 
# ACS total vs model-calculated total from prior base year run.
ACS_TOTAL_POPULATION        = 53953
MODEL_CALCULATED_POPULATION = 52788.25363
HOUSEHOLD_SIZE_ADJUSTMENT   = ACS_TOTAL_POPULATION / MODEL_CALCULATED_POPULATION

# Paths 
SCRIPTS_DIR = Path().absolute()                    # .../Base/scripts
DATA_DIR    = SCRIPTS_DIR.parents[0] / "data" / "raw_data"
OUT_DIR     = SCRIPTS_DIR.parents[0] / "data" / "processed_data"
LOOKUP_DIR  = SCRIPTS_DIR / "Lookup_Lists"
WORKSPACE   = Path(r"C:\GIS\Scratch.gdb")        # ArcGIS scratch workspace

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Checkpoint paths (Parquet â€” no pickle version lock-in) 
CKPT_SPATIAL_JOINS   = OUT_DIR / "parcel_spatial_joins.parquet"
CKPT_OCC_RATES       = OUT_DIR / "parcel_occupancy_rates.parquet"
CKPT_OCC_INTERP      = OUT_DIR / "parcel_occupancy_interpolated.parquet"
CKPT_SOCIOECONOMIC   = OUT_DIR / "parcel_socioeconomic.parquet"
CKPT_CAMPGROUND      = OUT_DIR / "campground_taz.parquet"
CKPT_SCHOOL          = OUT_DIR / "school_enrollment.parquet"
CKPT_OCC_RATES_TABLE = OUT_DIR / "occupancy_rates_table.parquet"

# Web service URLs 
BASE_URL = "https://maps.trpa.org/server/rest/services"
# REST endpoints for each layer used in the spatial joins and occupancy rate interpolation.
PARCEL_URL         = f"{BASE_URL}/Existing_Development/MapServer/2"
VHR_URL            = f"{BASE_URL}/VHR/MapServer/0"
TAZ_URL            = f"{BASE_URL}/Transportation_Planning/MapServer/6"
BLOCK_GROUPS_URL   = f"{BASE_URL}/Demographics/MapServer/27"
CENSUS_URL         = f"{BASE_URL}/Demographics/MapServer/28"
CAMPGROUND_URL     = f"{BASE_URL}/Recreation/MapServer/1"
CAMPVISITS_URL     = f"{BASE_URL}/Transportation_Planning/MapServer/14"
OCC_ZONES_URL      = f"{BASE_URL}/Transportation_Planning/MapServer/15"
OCC_RATES_URL      = f"{BASE_URL}/Transportation_Planning/MapServer/13"
SCHOOL_TABLE_URL   = f"{BASE_URL}/Transportation_Planning/MapServer/17"
SCHOOL_SPATIAL_URL = f"{BASE_URL}/Transportation_Planning/MapServer/16"

# Final parcel schema
FINAL_SCHEMA = [
    "APN", "Residential_Units", "TouristAccommodation_Units", "CommercialFloorArea_SqFt",
    "RoomsRented_PerDay", "VHR_Occupancy_Rate", "TAU_Occupancy_Rate",
    "PrimaryResidence_Rate", "SecondaryResidence_Rate",
    "HighIncome_Rate", "MediumIncome_Rate", "LowIncome_Rate",
    "PersonsPerUnit", "TAU_TYPE", "VHR", "BLOCK_GROUP", "TAZ",
    "OCCUPANCY_ZONE", "JURISDICTION", "COUNTY", "OWNERSHIP_TYPE",
    "EXISTING_LANDUSE", "WITHIN_TRPA_BNDY", "PARCEL_ACRES", "PARCEL_SQFT",
]

print("Config loaded.")
print(f"  Data dir  : {DATA_DIR}")
print(f"  Output dir: {OUT_DIR}")

## Imports

In [ ]:
import pandas as pd
import numpy as np
import os

# ArcGIS / spatial
import arcpy
from arcgis.features import FeatureLayer, GeoAccessor, GeoSeriesAccessor

# Local utilities
from utils import (
    get_fs_data, get_fs_data_query,
    get_fs_data_spatial, get_fs_data_spatial_query,
    # renamecolumns, update_if_contains,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

arcpy.env.workspace = str(WORKSPACE)
arcpy.env.outputCoordinateSystem = arcpy.SpatialReference("NAD 1983 UTM Zone 10N")
arcpy.env.overwriteOutput = True

print("Imports OK.")

## Helper Functions

In [ ]:
def _to_arcpy_geom(geom, sr):
    """Convert arcgis API Geometry to arcpy Geometry for InsertCursor compatibility."""
    if geom is None:
        return None
    if hasattr(geom, "as_arcpy"):
        return geom.as_arcpy
    return arcpy.FromWKT(geom.WKT, sr)


def _sdf_to_fc(sdf, fc_path, fields, shape_col):
    """Write SDF fields to a real feature class via arcpy InsertCursor (avoids arcgis API bugs)."""
    import os
    sdf_clean = sdf[fields + [shape_col]].dropna(subset=[shape_col]).copy()
    if sdf_clean.empty:
        raise ValueError(f"No valid geometries in SDF for fields {fields}")

    first_geom = sdf_clean[shape_col].iloc[0]
    geom_map   = {"point": "POINT", "multipoint": "MULTIPOINT",
                  "polygon": "POLYGON", "polyline": "POLYLINE"}
    geom_type  = geom_map.get(getattr(first_geom, "type", "point").lower(), "POINT")

    # spatialReference from arcgis API geometries is a dict {wkid:...}, not arcpy.SpatialReference
    sr_raw = getattr(first_geom, "spatialReference", None)
    if isinstance(sr_raw, arcpy.SpatialReference):
        sr = sr_raw
    elif isinstance(sr_raw, dict):
        wkid = sr_raw.get("wkid") or sr_raw.get("latestWkid")
        sr = arcpy.SpatialReference(int(wkid)) if wkid else arcpy.SpatialReference(4326)
    else:
        sr = arcpy.SpatialReference(4326)

    ws, name = os.path.split(fc_path)
    arcpy.management.CreateFeatureclass(ws, name, geom_type, spatial_reference=sr)

    dtype_map = {"int64": "LONG", "float64": "DOUBLE"}
    for fld in fields:
        ftype = dtype_map.get(str(sdf_clean[fld].dtype), "TEXT")
        if ftype == "TEXT":
            arcpy.management.AddField(fc_path, fld, "TEXT", field_length=255)
        else:
            arcpy.management.AddField(fc_path, fld, ftype)

    with arcpy.da.InsertCursor(fc_path, fields + ["SHAPE@"]) as cur:
        for _, row in sdf_clean.iterrows():
            arcpy_geom = _to_arcpy_geom(row[shape_col], sr)
            cur.insertRow([row[f] for f in fields] + [arcpy_geom])


def arcpy_spatial_join_attr(target_sdf, join_sdf, join_field,
                             match_option="HAVE_THEIR_CENTER_IN", key_field="APN"):
    """Spatial join using real feature classes in scratchGDB; returns dict {key_field: join_field value}.
    Uses unique FC names (uuid) per call to avoid schema-lock conflicts from prior failed runs."""
    import os, uuid
    uid       = uuid.uuid4().hex[:8]
    scratch   = arcpy.env.scratchGDB
    target_fc = os.path.join(scratch, f"target_sj_{uid}")
    join_fc   = os.path.join(scratch, f"join_sj_{uid}")
    out_fc    = os.path.join(scratch, f"out_sj_{uid}")

    t_shape = next((c for c in target_sdf.columns if c.upper() == "SHAPE"), None)
    j_shape = next((c for c in join_sdf.columns  if c.upper() == "SHAPE"), None)
    if t_shape is None:
        raise KeyError(f"No SHAPE column in target SDF. Columns: {list(target_sdf.columns)}")
    if j_shape is None:
        raise KeyError(f"No SHAPE column in join SDF. Columns: {list(join_sdf.columns)}")

    try:
        _sdf_to_fc(target_sdf, target_fc, [key_field],  t_shape)
        _sdf_to_fc(join_sdf,   join_fc,   [join_field], j_shape)

        arcpy.analysis.SpatialJoin(
            target_features=target_fc, join_features=join_fc, out_feature_class=out_fc,
            join_operation="JOIN_ONE_TO_ONE", join_type="KEEP_ALL",
            match_option=match_option
        )

        # Read result with cursor -- avoids arcgis from_featureclass bugs
        fc_fields = [f.name for f in arcpy.ListFields(out_fc)]
        fc_upper  = {f.upper(): f for f in fc_fields}
        kf_actual = fc_upper.get(key_field.upper())
        jf_actual = fc_upper.get(join_field.upper())
        if kf_actual is None or jf_actual is None:
            raise KeyError(
                f"Column not found in spatial join output. "
                f"Looking for: {key_field} (found={kf_actual is not None}), "
                f"{join_field} (found={jf_actual is not None}). "
                f"Available: {fc_fields}"
            )
        result = {}
        with arcpy.da.SearchCursor(out_fc, [kf_actual, jf_actual]) as cur:
            for row in cur:
                if row[1] is not None:
                    result[row[0]] = row[1]
        return result

    finally:
        # Always clean up scratch FCs
        for fc in [out_fc, target_fc, join_fc]:
            if arcpy.Exists(fc):
                arcpy.management.Delete(fc)


def map_rates(df, rate_df, zone_col, rate_col, target_col):
    """Map a rate from rate_df (keyed by zone_col) onto df, writing to target_col."""
    rate_map = rate_df.set_index(zone_col)[rate_col].to_dict()
    df[target_col] = df[zone_col].map(rate_map)
    return df


def check_dupes(df, col, drop=True):
    """Report and optionally remove duplicate rows by col."""
    dupes = df[df.duplicated(subset=col, keep=False)]
    if len(dupes):
        print(f"  WARNING: {len(dupes)} duplicate rows on {col}")
        if drop:
            df = df.drop_duplicates(subset=col, keep="first")
            print(f"  Dropped. Remaining: {len(df)}")
    else:
        print(f"  No duplicates on {col}.")
    return df

def qa_summary(df, label, unit_cols=None):
    '''Print a concise QA snapshot: row count and column sums for unit_cols.'''
    print(f'=== QA: {label} ===')
    print(f'  Rows: {len(df):,}')
    if unit_cols:
        for col in unit_cols:
            if col in df.columns:
                print(f'  {col}: {df[col].fillna(0).sum():,.0f}')
            else:
                print(f'  {col}: (column not found)')


---
## Stage 1 - Get Data
Fetch all source datasets from TRPA web services.

**Outputs:** Raw DataFrames in memory for downstream stages

> If a service URL returns an error, correct the URL in Config and re-run this cell.

In [ ]:
print("Fetching parcels...")
sdfParcel_raw = get_fs_data_spatial_query(PARCEL_URL, query_params=f"Year = {PARCEL_YEAR}")
print(f"  Parcels        : {len(sdfParcel_raw):,}")

print("Fetching VHR parcel list...")
df_vhr_list = get_fs_data(VHR_URL)
print(f"  VHR records    : {len(df_vhr_list):,}")

print("Fetching TAZ...")
sdf_taz = get_fs_data_spatial(TAZ_URL)
print(f"  TAZ zones      : {len(sdf_taz):,}")

print("Fetching block groups...")
sdf_block_groups = get_fs_data_spatial(BLOCK_GROUPS_URL)
print(f"  Block groups   : {len(sdf_block_groups):,}")

print("Fetching census demographics...")
df_census = get_fs_data_query(CENSUS_URL, query_params=f"year_sample = {CENSUS_YEAR}")
print(f"  Census records : {len(df_census):,}")

print("Fetching occupancy zones...")
sdf_occ_zones = get_fs_data_spatial(OCC_ZONES_URL)
print(f"  Occ zones      : {len(sdf_occ_zones):,}")

print("Fetching occupancy rates...")
df_occ_rates_raw = get_fs_data(OCC_RATES_URL)
print(f"  Occ rate rows  : {len(df_occ_rates_raw):,}")

print("Fetching campgrounds...")
sdf_campgrounds = get_fs_data_spatial(CAMPGROUND_URL)
df_campvisits   = get_fs_data(CAMPVISITS_URL)
print(f"  Campgrounds    : {len(sdf_campgrounds):,}")

print("Fetching schools...")
df_school_table = get_fs_data_query(SCHOOL_TABLE_URL, query_params=f"Year = '{SCHOOL_YEAR}'")
sdf_schools     = get_fs_data_spatial(SCHOOL_SPATIAL_URL)
print(f"  School records : {len(df_school_table):,}")

print("\nAll services fetched.")

---
## Stage 2 - Parcel Classification
Apply unit changes, assign VHR flag and TAU type, zero closed parcels,
then spatial-join to TAZ / block group / occupancy zone.

**Inputs:** `sdfParcel_raw`, `df_vhr_list`, `sdf_taz`, `sdf_block_groups`, `sdf_occ_zones`
**Checkpoint:** `parcel_spatial_joins.parquet`

### 2a - Apply known unit changes (development_2022_2025)

In [ ]:
df_dev = (
    pd.read_csv(LOOKUP_DIR / "development_2022_2025.csv")
    [["Fixed_APN", "Change"]]
    .rename(columns={"Fixed_APN": "APN"})
    .dropna(subset=["Change"])
    .astype({"Change": int})
)

sdfParcel = sdfParcel_raw.copy()
sdfParcel = sdfParcel.merge(df_dev, on="APN", how="left")
sdfParcel["Change"] = sdfParcel["Change"].fillna(0).astype(int)
sdfParcel["Residential_Units_Adjusted"] = (
    sdfParcel["Residential_Units"].fillna(0).astype(int) + sdfParcel["Change"]
).clip(lower=0)

changed = (sdfParcel["Change"] != 0).sum()
print(f"Unit changes applied to {changed} parcels")
print(f"Residential_Units_Adjusted total: {sdfParcel['Residential_Units_Adjusted'].sum():,}")

### 2b - VHR flag

In [ ]:
vhr_apns = set(df_vhr_list["APN"].dropna().astype(str))
sdfParcel["VHR"] = sdfParcel["APN"].astype(str).isin(vhr_apns).map({True: "Yes", False: "No"})
print(f"VHR parcels : {(sdfParcel['VHR'] == 'Yes').sum():,}")
print(f"Non-VHR     : {(sdfParcel['VHR'] == 'No').sum():,}")

### 2c - TAU type classification

In [ ]:
df_tau_lookup = pd.read_csv(LOOKUP_DIR / "lookup_tau_type.csv")[["APN", "TAU_Type"]]
tau_type_map  = df_tau_lookup.set_index("APN")["TAU_Type"].to_dict()

# Default: no TAU units -> N/A; has TAU units -> HotelMotel; lookup overrides with Casino/Resort
sdfParcel["TAU_TYPE"] = "N/A"
has_tau = sdfParcel["TouristAccommodation_Units"].fillna(0) > 0
sdfParcel.loc[has_tau, "TAU_TYPE"] = "HotelMotel"
sdfParcel["TAU_TYPE"] = sdfParcel["APN"].map(tau_type_map).fillna(sdfParcel["TAU_TYPE"])

print("TAU_TYPE distribution:")
print(sdfParcel["TAU_TYPE"].value_counts().to_string())

### 2d - Zero out closed tourist parcels

In [ ]:
df_closed   = pd.read_csv(LOOKUP_DIR / "closed_tourist_parcels.csv")
closed_apns = set(df_closed["APN"].astype(str))
n_closed    = sdfParcel["APN"].astype(str).isin(closed_apns).sum()
sdfParcel.loc[sdfParcel["APN"].astype(str).isin(closed_apns), "TouristAccommodation_Units"] = 0
print(f"Closed tourist parcels zeroed: {n_closed}")

### 2e - Spatial joins: TAZ, block group, occupancy zone  [ARCPY]

In [ ]:
# Spatial joins: TAZ, block groups, occupancy zones
print("Joining TAZ...")
sdfParcel["TAZ"] = sdfParcel["APN"].map(
    arcpy_spatial_join_attr(sdfParcel, sdf_taz, "TAZ")
)

print("Joining block groups...")
sdfParcel["BLOCK_GROUP"] = sdfParcel["APN"].map(
    arcpy_spatial_join_attr(sdfParcel, sdf_block_groups, "TRPAID")
)

print("Joining occupancy zones...")
sdfParcel["OCCUPANCY_ZONE"] = sdfParcel["APN"].map(
    arcpy_spatial_join_attr(sdfParcel, sdf_occ_zones, "OccupancyRate_ZoneID")
)

# CSLT VHR parcels use a combined zone for occupancy rate averaging
cslt_vhr = (sdfParcel["JURISDICTION"] == "CSLT") & (sdfParcel["VHR"] == "Yes")
sdfParcel.loc[cslt_vhr, "OCCUPANCY_ZONE"] = "CSLT_ALL"
print(f"CSLT_ALL override applied to {cslt_vhr.sum()} parcels")

### 2f - Trim to final schema and save checkpoint

In [ ]:
for field in FINAL_SCHEMA:
    if field not in sdfParcel.columns:
        sdfParcel[field] = np.nan

keep_cols = FINAL_SCHEMA + ["Residential_Units_Adjusted", "Change"]
sdfParcel = sdfParcel[[c for c in keep_cols if c in sdfParcel.columns]].copy()

sdfParcel.to_parquet(CKPT_SPATIAL_JOINS, index=False)
print(f"Checkpoint saved: {CKPT_SPATIAL_JOINS}")

### QA - Stage 2

In [ ]:
qa_summary(sdfParcel, "After Spatial Joins",
           unit_cols=["Residential_Units", "Residential_Units_Adjusted", "TouristAccommodation_Units"])
print(f"  TAZ nulls        : {sdfParcel['TAZ'].isna().sum()}")
print(f"  BLOCK_GROUP nulls: {sdfParcel['BLOCK_GROUP'].isna().sum()}")
print(f"  OCC_ZONE nulls   : {sdfParcel['OCCUPANCY_ZONE'].isna().sum()}")
sdfParcel[["APN","TAZ","BLOCK_GROUP","OCCUPANCY_ZONE","VHR","TAU_TYPE"]].head()

---
## Stage 3 - Occupancy Rate Application
Average zone-level rates across configured timeframes, then map onto parcels.

**Inputs:** `parcel_spatial_joins.parquet`, `df_occ_rates_raw`
**Checkpoints:** `occupancy_rates_table.parquet`, `parcel_occupancy_rates.parquet`

In [ ]:
sdfParcel = pd.read_parquet(CKPT_SPATIAL_JOINS)
print(f"Loaded {len(sdfParcel):,} parcels from checkpoint")

In [ ]:
df_occ = df_occ_rates_raw.copy()

# Build a Date string from Year + MonthNum (monthly rows only)
df_occ_monthly = df_occ[df_occ["MonthNum"].notna()].copy()
df_occ_monthly["Date"] = pd.to_datetime({
    "year": df_occ_monthly["Year"].astype(int),
    "month": df_occ_monthly["MonthNum"].astype(int),
    "day": 1
}).dt.strftime("%Y-%m-%d")

# Filter to selected timeframes
df_occ_monthly = df_occ_monthly[df_occ_monthly["Date"].isin(OCCUPANCY_TIMEFRAMES)]
print(f"Rate rows after timeframe filter: {len(df_occ_monthly):,}")
print(f"Type values present: {sorted(df_occ_monthly['Type'].dropna().unique())}")

# TAU = hotel/casino/resort types; VHR = vacation home rentals
TAU_TYPES = ["HotelMotel", "Casino", "Resort"]

df_tau = (
    df_occ_monthly[df_occ_monthly["Type"].isin(TAU_TYPES)]
    .groupby("ZoneID", as_index=False)["OccupancyRate"].mean()
    .rename(columns={"ZoneID": "OCCUPANCY_ZONE", "OccupancyRate": "TAU_Occupancy_Rate"})
)
df_vhr = (
    df_occ_monthly[df_occ_monthly["Type"] == "VHR"]
    .groupby("ZoneID", as_index=False)["OccupancyRate"].mean()
    .rename(columns={"ZoneID": "OCCUPANCY_ZONE", "OccupancyRate": "VHR_Occupancy_Rate"})
)

# Merge TAU and VHR averaged rates per zone
df_occ_avg = df_tau.merge(df_vhr, on="OCCUPANCY_ZONE", how="outer")
df_occ_avg.to_parquet(CKPT_OCC_RATES_TABLE, index=False)
print(f"Zones with averaged rates: {len(df_occ_avg)}")
df_occ_avg

In [ ]:
# Map averaged zone rates onto parcels
sdfParcel = map_rates(sdfParcel, df_occ_avg, "OCCUPANCY_ZONE", "TAU_Occupancy_Rate", "TAU_Occupancy_Rate")
sdfParcel = map_rates(sdfParcel, df_occ_avg, "OCCUPANCY_ZONE", "VHR_Occupancy_Rate", "VHR_Occupancy_Rate")

# Zero rates where there are no relevant units
sdfParcel.loc[sdfParcel["TouristAccommodation_Units"].fillna(0) == 0, "TAU_Occupancy_Rate"] = 0
sdfParcel.loc[sdfParcel["VHR"] == "No",                               "VHR_Occupancy_Rate"] = 0

sdfParcel.to_parquet(CKPT_OCC_RATES, index=False)
print(f"Checkpoint saved: {CKPT_OCC_RATES}")

### QA - Stage 3

In [ ]:
tau_p = sdfParcel[sdfParcel["TouristAccommodation_Units"].fillna(0) > 0]
vhr_p = sdfParcel[sdfParcel["VHR"] == "Yes"]
print(f"TAU parcels missing rate: {tau_p['TAU_Occupancy_Rate'].isna().sum()}")
print(f"VHR parcels missing rate: {vhr_p['VHR_Occupancy_Rate'].isna().sum()}")
print(f"TAU rate range: {sdfParcel['TAU_Occupancy_Rate'].min():.3f} to {sdfParcel['TAU_Occupancy_Rate'].max():.3f}")
print(f"VHR rate range: {sdfParcel['VHR_Occupancy_Rate'].min():.3f} to {sdfParcel['VHR_Occupancy_Rate'].max():.3f}")

---
## Stage 4 - IDW Interpolation of Missing Occupancy Rates
For parcels with TAU or VHR units but no zone rate, use IDW on neighbouring
known-rate parcels. Any still-null rates after IDW get the basin-wide fallback.

**Inputs:** `parcel_occupancy_rates.parquet`
**Checkpoint:** `parcel_occupancy_interpolated.parquet`

In [ ]:
# Requires ArcGIS Pro + Spatial Analyst extension
from arcpy.sa import Idw
arcpy.CheckOutExtension("Spatial")
sdfParcel = pd.read_parquet(CKPT_OCC_RATES)
# Re-attach geometry for spatial operations
sdfParcel_geo = get_fs_data_spatial_query(PARCEL_URL, query_params=f"Year = {PARCEL_YEAR}")
sdfParcel = sdfParcel.merge(sdfParcel_geo[["APN", "SHAPE"]], on="APN", how="left")
print(f"Loaded {len(sdfParcel):,} parcels with geometry")

In [ ]:
# IDW interpolation function
def idw_fill_nulls(sdf, rate_col, unit_col=None, unit_threshold=0, filter_mask=None):
    '''IDW-interpolate rate_col for parcels where it is null.

    Parcels to interpolate are selected by either:
      - filter_mask  : a boolean Series (takes priority when provided), or
      - unit_col > unit_threshold  : legacy numeric-unit filter.
    '''
    if filter_mask is not None:
        base_mask = filter_mask
    else:
        base_mask = sdf[unit_col].fillna(0) > unit_threshold

    needs_fill = sdf[rate_col].isna() & base_mask
    has_rate   = sdf[rate_col].notna() & base_mask

    if needs_fill.sum() == 0:
        print(f'  {rate_col}: no nulls to fill')
        return sdf
    print(f'  {rate_col}: filling {needs_fill.sum()} null(s) via IDW')
    source_fc = r'memory\idw_source'
    target_fc = r'memory\idw_target'
    out_tbl   = r'memory\idw_values'
    sdf[has_rate][['APN', rate_col, 'SHAPE']].spatial.to_featureclass(source_fc)
    sdf[needs_fill][['APN', 'SHAPE']].spatial.to_featureclass(target_fc)
    idw_raster = Idw(source_fc, rate_col, cell_size=100, power=2)
    arcpy.sa.ExtractValuesToPoints(target_fc, idw_raster, out_tbl)
    result = pd.DataFrame.spatial.from_featureclass(out_tbl)[['APN', 'RASTERVALU']]
    result = result.rename(columns={'RASTERVALU': '_interp'})
    sdf = sdf.merge(result, on='APN', how='left')
    fill_mask = sdf[rate_col].isna() & sdf['_interp'].notna()
    sdf.loc[fill_mask, rate_col] = sdf.loc[fill_mask, '_interp']
    sdf = sdf.drop(columns=['_interp'])
    remaining = sdf[rate_col].isna() & base_mask
    print(f'  {rate_col}: {remaining.sum()} still null after IDW')
    return sdf


# TAU: fill nulls for parcels that have tourist accommodation units
sdfParcel = idw_fill_nulls(sdfParcel, 'TAU_Occupancy_Rate',
                           unit_col='TouristAccommodation_Units')

# VHR: fill nulls for VHR-flagged parcels (mirrors notebook.ipynb — VHR == 'Yes')
sdfParcel = idw_fill_nulls(sdfParcel, 'VHR_Occupancy_Rate',
                           filter_mask=(sdfParcel['VHR'] == 'Yes'))


In [ ]:
# Apply basin-wide fallbacks for any remaining nulls
tau_null = sdfParcel["TAU_Occupancy_Rate"].isna() & (sdfParcel["TouristAccommodation_Units"].fillna(0) > 0)
vhr_null = sdfParcel["VHR_Occupancy_Rate"].isna() & (sdfParcel["VHR"] == "Yes")
sdfParcel.loc[tau_null, "TAU_Occupancy_Rate"] = DEFAULT_TAU_OCCUPANCY
sdfParcel.loc[vhr_null, "VHR_Occupancy_Rate"] = DEFAULT_VHR_OCCUPANCY
print(f"Fallback TAU ({DEFAULT_TAU_OCCUPANCY}) applied to {tau_null.sum()} parcels")
print(f"Fallback VHR ({DEFAULT_VHR_OCCUPANCY}) applied to {vhr_null.sum()} parcels")

sdfParcel.drop(columns=["SHAPE"], errors="ignore").to_parquet(CKPT_OCC_INTERP, index=False)
print(f"Checkpoint saved: {CKPT_OCC_INTERP}")

### QA - Stage 4

In [ ]:
tau_p = sdfParcel[sdfParcel["TouristAccommodation_Units"].fillna(0) > 0]
vhr_p = sdfParcel[sdfParcel["VHR"] == "Yes"]
print(f"TAU parcels still missing rate: {tau_p['TAU_Occupancy_Rate'].isna().sum()}")
print(f"VHR parcels still missing rate: {vhr_p['VHR_Occupancy_Rate'].isna().sum()}")
print("\nTAU rate distribution:")
print(tau_p["TAU_Occupancy_Rate"].describe().round(3).to_string())

---
## Stage 5 - Census / Socioeconomic Attribution
Map block-group demographics (occupancy, seasonal rate, household size, income)
onto individual parcels, then derive unit-count fields.

**Inputs:** `parcel_occupancy_interpolated.parquet`, `df_census`
**Checkpoint:** `parcel_socioeconomic.parquet`

In [ ]:
sdfParcel = pd.read_parquet(CKPT_OCC_INTERP)
print(f"Loaded {len(sdfParcel):,} parcels from checkpoint")

### 5a - Occupancy and seasonal rates from census

In [ ]:
df_cen = df_census.copy()
df_cen["TRPAID"] = df_cen["TRPAID"].astype(str).str.zfill(16)

# B25002_002E = occupied units, B25002_003E = vacant, B25004_006E = vacant for seasonal use
occ_vars  = ["B25002_002E", "B25002_003E", "B25004_006E"]
df_occ_bg = (
    df_cen[df_cen["variable_code"].isin(occ_vars)]
    .pivot_table(index="TRPAID", columns="variable_code", values="value", aggfunc="first")
    .reset_index()
)
total_units = df_occ_bg["B25002_002E"].fillna(0) + df_occ_bg["B25002_003E"].fillna(0)
df_occ_bg["PrimaryResidence_Rate"]   = df_occ_bg["B25002_002E"].fillna(0) / total_units.replace(0, np.nan)
df_occ_bg["SecondaryResidence_Rate"] = (
    df_occ_bg["B25004_006E"].fillna(0) / df_occ_bg["B25002_003E"].fillna(0).replace(0, np.nan)
)
print(f"Occupancy by block group: {len(df_occ_bg)} rows")

### 5b - Household size

In [ ]:
# B25010_001E = average household size of occupied housing units
df_hh = (
    df_cen[df_cen["variable_code"] == "B25010_001E"][["TRPAID", "value"]]
    .rename(columns={"value": "PersonsPerUnit_Raw"})
    .copy()
)
df_hh["TRPAID"]         = df_hh["TRPAID"].astype(str).str.zfill(16)
df_hh["PersonsPerUnit"] = df_hh["PersonsPerUnit_Raw"] * HOUSEHOLD_SIZE_ADJUSTMENT
print(f"Household size records : {len(df_hh)}")
print(f"Adjustment factor      : {HOUSEHOLD_SIZE_ADJUSTMENT:.6f}")

### 5c - Income distribution

In [ ]:
df_income_codes = pd.read_csv(LOOKUP_DIR / "income_census_codes.csv")
df_inc = df_cen[df_cen["variable_code"].isin(df_income_codes["variable_code"])].copy()
df_inc["TRPAID"] = df_inc["TRPAID"].astype(str).str.zfill(16)
df_inc = df_inc.merge(df_income_codes[["variable_code", "category"]], on="variable_code", how="left")

df_inc_pivot = (
    df_inc.groupby(["TRPAID", "category"])["value"]
    .sum().unstack(fill_value=0).reset_index()
)
inc_total = df_inc_pivot[["High Income", "Medium Income", "Low Income"]].sum(axis=1)
df_inc_pivot["HighIncome_Rate"]   = df_inc_pivot["High Income"]   / inc_total.replace(0, np.nan)
df_inc_pivot["MediumIncome_Rate"] = df_inc_pivot["Medium Income"] / inc_total.replace(0, np.nan)
df_inc_pivot["LowIncome_Rate"]    = df_inc_pivot["Low Income"]    / inc_total.replace(0, np.nan)
print(f"Income distribution by block group: {len(df_inc_pivot)} rows")

### 5d - Map to parcels and derive unit counts

In [ ]:
# Combine all block-group attribute tables
df_bg = (
    df_occ_bg[["TRPAID", "PrimaryResidence_Rate", "SecondaryResidence_Rate"]]
    .merge(df_hh[["TRPAID", "PersonsPerUnit"]], on="TRPAID", how="outer")
    .merge(df_inc_pivot[["TRPAID", "HighIncome_Rate", "MediumIncome_Rate", "LowIncome_Rate"]], on="TRPAID", how="outer")
)

sdfParcel["BLOCK_GROUP"] = sdfParcel["BLOCK_GROUP"].astype(str).str.zfill(16)
sdfParcel = sdfParcel.merge(df_bg, left_on="BLOCK_GROUP", right_on="TRPAID", how="left", suffixes=("", "_census"))

# Use census values, falling back to any pre-existing parcel values
for col in ["PrimaryResidence_Rate", "SecondaryResidence_Rate", "PersonsPerUnit",
            "HighIncome_Rate", "MediumIncome_Rate", "LowIncome_Rate"]:
    census_col = col + "_census"
    if census_col in sdfParcel.columns:
        sdfParcel[col] = sdfParcel[census_col].combine_first(sdfParcel[col])
        sdfParcel = sdfParcel.drop(columns=[census_col])

# Derive unit count fields
res = sdfParcel["Residential_Units_Adjusted"].fillna(0)
sdfParcel["OccupiedUnits"]   = res * sdfParcel["PrimaryResidence_Rate"].fillna(0)
sdfParcel["UnoccupiedUnits"] = res - sdfParcel["OccupiedUnits"]
sdfParcel["SeasonalUnits"]   = sdfParcel["UnoccupiedUnits"] * sdfParcel["SecondaryResidence_Rate"].fillna(0)
sdfParcel["HighUnits"]       = sdfParcel["OccupiedUnits"] * sdfParcel["HighIncome_Rate"].fillna(0)
sdfParcel["MediumUnits"]     = sdfParcel["OccupiedUnits"] * sdfParcel["MediumIncome_Rate"].fillna(0)
sdfParcel["LowUnits"]        = sdfParcel["OccupiedUnits"] * sdfParcel["LowIncome_Rate"].fillna(0)
sdfParcel["People"]          = sdfParcel["OccupiedUnits"] * sdfParcel["PersonsPerUnit"].fillna(0)

sdfParcel.drop(columns=["TRPAID"], errors="ignore", inplace=True)

### 5e - Rounding and integer validation

In [ ]:
# â”€â”€ Step 1: save unrounded income values before rounding (needed for tie-break fix) 
high_unrounded   = sdfParcel["HighUnits"].copy()
medium_unrounded = sdfParcel["MediumUnits"].copy()
low_unrounded    = sdfParcel["LowUnits"].copy()

# â”€â”€ Step 2: round occupied units and income-level units to nearest integer
for col in ["OccupiedUnits", "HighUnits", "MediumUnits", "LowUnits"]:
    sdfParcel[col] = sdfParcel[col].fillna(0).round(0).astype(int)

# Step 3: if rounded OccupiedUnits > 0 but income units sum to 0,
#           set the income class with the highest unrounded value to 1 
income_sum = sdfParcel["HighUnits"] + sdfParcel["MediumUnits"] + sdfParcel["LowUnits"]
fix_income = (sdfParcel["OccupiedUnits"] > 0) & (income_sum == 0)
if fix_income.sum() > 0:
    print(f"  Income fix applied to {fix_income.sum()} parcels")
    best_class = pd.DataFrame({
        "HighUnits":   high_unrounded[fix_income],
        "MediumUnits": medium_unrounded[fix_income],
        "LowUnits":    low_unrounded[fix_income],
    }).idxmax(axis=1)
    for idx, cls in best_class.items():
        sdfParcel.loc[idx, cls] = 1
else:
    print("  Income fix: no parcels affected")

# Step 4: round People to nearest integer
sdfParcel["People"] = sdfParcel["People"].fillna(0).round(0).astype(int)

# Step 5: if People > 0 but OccupiedUnits == 0, zero out People
pop_fix = (sdfParcel["People"] > 0) & (sdfParcel["OccupiedUnits"] == 0)
if pop_fix.sum() > 0:
    print(f"  Population zeroed for {pop_fix.sum()} parcels with 0 occupied units")
    sdfParcel.loc[pop_fix, "People"] = 0
else:
    print("  Population fix: no parcels affected")

sdfParcel.to_parquet(CKPT_SOCIOECONOMIC, index=False)
print(f"Checkpoint saved: {CKPT_SOCIOECONOMIC}")

### QA - Stage 5

In [ ]:
qa_summary(sdfParcel, "After Socioeconomic Attribution",
    unit_cols=["Residential_Units_Adjusted", "OccupiedUnits", "SeasonalUnits", "People"])
print(f"  PrimaryResidence_Rate nulls  : {sdfParcel['PrimaryResidence_Rate'].isna().sum()}")
print(f"  SecondaryResidence_Rate nulls: {sdfParcel['SecondaryResidence_Rate'].isna().sum()}")
print(f"  PersonsPerUnit nulls         : {sdfParcel['PersonsPerUnit'].isna().sum()}")

---
## Stage 6 - Campground Processing
Aggregate campground overnight capacity to TAZ level.

**Inputs:** `sdf_campgrounds`, `df_campvisits`
**Checkpoint:** `campground_taz.parquet`

In [ ]:
# [ARCPY] Spatial join campground locations to TAZ
camp_taz_lookup = arcpy_spatial_join_attr(
    sdf_campgrounds, sdf_taz, "TAZ", key_field="RECREATION_NAME"
)
sdf_campgrounds["TAZ"] = sdf_campgrounds["RECREATION_NAME"].map(camp_taz_lookup)
print(f"Campground TAZ nulls: {sdf_campgrounds['TAZ'].isna().sum()}")

In [ ]:
df_camp = sdf_campgrounds.merge(df_campvisits, left_on="RECREATION_NAME", right_on="Campground", how="left")
df_camp = df_camp[df_camp["Campground"] != "Bayview Campground"].copy()

# Compute sites sold: Total_Sites * Occupancy_Rate (mirrors notebook.ipynb)
df_camp["SitesSold"] = df_camp["Total_Sites"] * df_camp["Occupancy_Rate"]

df_camp_taz = (
    df_camp.groupby("TAZ", as_index=False)
    .agg(Total_Sites=("Total_Sites", "sum"),
         campground  =("SitesSold",  "sum"))   # campground = sites sold (occupied)
)

df_camp_taz.to_parquet(CKPT_CAMPGROUND, index=False)
print(f"TAZs with campgrounds : {len(df_camp_taz)}")
print(f"Total campground sites: {df_camp_taz['Total_Sites'].sum():,}")
print(f"Total sites sold      : {df_camp_taz['campground'].sum():,.0f}")


---
## Stage 7 - School Enrollment
Spatial-join schools to TAZ, then aggregate enrollment counts.

**Inputs:** `df_school_table`, `sdf_schools`, `sdf_taz`
**Checkpoint:** `school_enrollment.parquet`  |  **Output:** `SchoolEnrollment.csv`

In [ ]:
# [ARCPY] Spatial join school locations to TAZ (key on NAME to match reference)
school_taz_lookup = arcpy_spatial_join_attr(sdf_schools, sdf_taz, "TAZ", key_field="NAME")

# Classify each school by type based on its name (mirrors notebook.ipynb logic)
sdf_schools_typed = sdf_schools.copy()
sdf_schools_typed["TYPE"] = None
sdf_schools_typed.loc[sdf_schools_typed["NAME"].str.contains("elementary", case=False), "TYPE"] = "Elementary School"
sdf_schools_typed.loc[sdf_schools_typed["NAME"].str.contains("middle",     case=False), "TYPE"] = "Middle School"
sdf_schools_typed.loc[sdf_schools_typed["NAME"].str.contains("high",       case=False), "TYPE"] = "High School"
sdf_schools_typed.loc[sdf_schools_typed["NAME"].str.contains("college",    case=False), "TYPE"] = "College"
# Default unclassified schools to Elementary School
sdf_schools_typed.loc[sdf_schools_typed["TYPE"].isnull(), "TYPE"] = "Elementary School"

# Map TAZ onto each school point; normalise to numeric to match sdf_taz dtype
sdf_schools_typed["TAZ"] = pd.to_numeric(
    sdf_schools_typed["NAME"].map(school_taz_lookup), errors="coerce"
)

school_grouped = (
    sdf_schools_typed.dropna(subset=["TAZ"])
    .groupby(["TYPE", "TAZ"], as_index=False)["ENROLLMENT"]
    .sum()
)

# Pivot: one row per TAZ, one column per school type
school_pivot = school_grouped.pivot(index="TAZ", columns="TYPE", values="ENROLLMENT").reset_index()
school_pivot.columns.name = None
school_pivot["TAZ"] = pd.to_numeric(school_pivot["TAZ"], errors="coerce")

# Merge with full TAZ list (cast both keys to numeric to avoid dtype conflict)
taz_frame = sdf_taz[["TAZ"]].copy()
taz_frame["TAZ"] = pd.to_numeric(taz_frame["TAZ"], errors="coerce")
df_school_taz = pd.merge(taz_frame, school_pivot, on="TAZ", how="left")

# Rename to standard column names
df_school_taz.rename(columns={
    "Elementary School": "elementary_school_enrollment",
    "Middle School":     "middle_school_enrollment",
    "High School":       "high_school_enrollment",
    "College":           "college_enrollment",
}, inplace=True)

# Ensure all four columns exist even if no schools of that type were found
for col in ["elementary_school_enrollment", "middle_school_enrollment",
            "high_school_enrollment", "college_enrollment"]:
    if col not in df_school_taz.columns:
        df_school_taz[col] = 0

enroll_cols = ["elementary_school_enrollment", "middle_school_enrollment",
               "high_school_enrollment", "college_enrollment"]

df_school_taz = (
    df_school_taz.groupby("TAZ", as_index=False)[enroll_cols]
    .sum()
    .fillna(0)
    .astype({c: int for c in enroll_cols})
)

df_school_taz.to_parquet(CKPT_SCHOOL, index=False)
df_school_taz.to_csv(OUT_DIR / "SchoolEnrollment.csv", index=False)
print(f"School checkpoint saved.")
print(f"Total enrollment: {df_school_taz[enroll_cols].sum().sum():,}")


---
## Stage 8 - TAZ Aggregation & Output CSV Export
Load all checkpoints and build the five travel-demand model input files.

| Output file | Contents |
|---|---|
| `OvernightVisitorZonalData_Summer.csv` | TAU rooms by type, campground sites, % seasonal |
| `VisitorOccupancyRates_Summer.csv`     | Weighted TAU and VHR occupancy rates by TAZ |
| `SocioEcon_Summer.csv`                 | Occupancy, seasonal, household size, income shares |
| `Employment.csv`                       | Employment by sector and TAZ |
| `inputs_summarized.csv`                | All of the above joined into one wide table |

### 8a - Load checkpoints

In [ ]:
sdfParcel   = pd.read_parquet(CKPT_SOCIOECONOMIC)
df_camp_taz = pd.read_parquet(CKPT_CAMPGROUND)
df_school   = pd.read_parquet(CKPT_SCHOOL)
print(f"Parcels      : {len(sdfParcel):,}")
print(f"Camp TAZ rows: {len(df_camp_taz)}")
print(f"School rows  : {len(df_school)}")

### 8b - Overnight visitor zonal data

In [ ]:
# TAU rooms by type aggregated to TAZ
df_tau_taz = (
    sdfParcel[sdfParcel["TouristAccommodation_Units"].fillna(0) > 0]
    .groupby(["TAZ", "TAU_TYPE"])["TouristAccommodation_Units"]
    .sum().unstack(fill_value=0).reset_index()
)
for col in ["HotelMotel", "Casino", "Resort"]:
    if col not in df_tau_taz.columns:
        df_tau_taz[col] = 0
df_tau_taz = df_tau_taz.rename(columns={"HotelMotel": "hotelmotel", "Casino": "casino", "Resort": "resort"})

# Seasonal percentage by TAZ
df_seasonal = (
    sdfParcel.groupby("TAZ", as_index=False)
    .agg(seasonal=("SeasonalUnits", "sum"), total_res=("Residential_Units_Adjusted", "sum"))
)
df_seasonal["percentHouseSeasonal"] = (
    df_seasonal["seasonal"] / df_seasonal["total_res"].replace(0, np.nan)
).fillna(0)

df_overnight = (
    df_tau_taz
    .merge(df_camp_taz, on="TAZ", how="outer")
    .merge(df_seasonal[["TAZ", "percentHouseSeasonal"]], on="TAZ", how="outer")
    .fillna(0)
)
for col in ["hotelmotel", "casino", "resort", "campground"]:
    if col in df_overnight.columns:
        df_overnight[col] = df_overnight[col].astype(int)

df_overnight = df_overnight[["TAZ","hotelmotel","casino","resort","campground","percentHouseSeasonal"]]
df_overnight.to_csv(OUT_DIR / "OvernightVisitorZonalData_Summer.csv", index=False)
print(f"OvernightVisitorZonalData_Summer.csv -- {len(df_overnight)} TAZs")
df_overnight.head()

### 8c - Visitor occupancy rates

In [ ]:
# TAU: weighted average by unit count per TAZ
tau_sdf = sdfParcel[sdfParcel["TouristAccommodation_Units"].fillna(0) > 0].copy()
tau_sdf["tau_weighted"] = tau_sdf["TAU_Occupancy_Rate"] * tau_sdf["TouristAccommodation_Units"]
df_tau_occ = (
    tau_sdf.groupby("TAZ")
    .agg(tau_sum=("tau_weighted", "sum"), tau_units=("TouristAccommodation_Units", "sum"))
    .assign(tau_occupancy_rate=lambda d: d["tau_sum"] / d["tau_units"].replace(0, np.nan))
    .reset_index()[["TAZ", "tau_occupancy_rate"]]
)

# VHR: simple mean per TAZ
df_vhr_occ = (
    sdfParcel[sdfParcel["VHR"] == "Yes"]
    .groupby("TAZ", as_index=False)
    .agg(vhr_occupancy_rate=("VHR_Occupancy_Rate", "mean"))
)

df_visitor_occ = df_tau_occ.merge(df_vhr_occ, on="TAZ", how="outer").fillna(0)
df_visitor_occ.to_csv(OUT_DIR / "VisitorOccupancyRates_Summer.csv", index=False)
print(f"VisitorOccupancyRates_Summer.csv -- {len(df_visitor_occ)} TAZs")
df_visitor_occ.head()

### 8d - Socioeconomic data

In [ ]:
df_socio = (
    sdfParcel.groupby("TAZ", as_index=False)
    .agg(
        total_residential_units  =("Residential_Units_Adjusted", "sum"),
        occupied_units           =("OccupiedUnits", "sum"),
        occupancy_rate           =("PrimaryResidence_Rate", "mean"),
        seasonal_rate            =("SecondaryResidence_Rate", "mean"),
        household_size           =("PersonsPerUnit", "mean"),
        high_income_proportion   =("HighIncome_Rate", "mean"),
        middle_income_proportion =("MediumIncome_Rate", "mean"),
        low_income_proportion    =("LowIncome_Rate", "mean"),
        people                   =("People", "sum"),
    )
    .fillna(0)
)
df_socio.to_csv(OUT_DIR / "SocioEcon_Summer.csv", index=False)
print(f"SocioEcon_Summer.csv -- {len(df_socio)} TAZs")
df_socio.head()

### 8e - Employment
Employment is carried forward unchanged from the 2022 base year.
The source file lives in the same `processed_data` folder from the prior run.
Update `EMPLOYMENT_SOURCE` in Config if the path ever changes.

In [ ]:
# Employment does not change from the 2022 base year copy it directly.
# Source: processed_data from the 2022 base year run (same folder).
employment_source = OUT_DIR / "Employment.csv"

if employment_source.exists():
    df_employ = pd.read_csv(employment_source)
    # Write back to OUT_DIR (no-op if already there, ensures file is present
    # even if OUT_DIR was cleared between runs)
    df_employ.to_csv(OUT_DIR / "Employment.csv", index=False)
    print(f"Employment.csv copied from 2022 base year -- {len(df_employ)} TAZs")
    df_employ.head()
else:
    # Fallback: look for it in the raw_data folder
    fallback = DATA_DIR / "Employment.csv"
    if fallback.exists():
        df_employ = pd.read_csv(fallback)
        df_employ.to_csv(OUT_DIR / "Employment.csv", index=False)
        print(f"Employment.csv copied from raw_data -- {len(df_employ)} TAZs")
    else:
        print(f"WARNING: Employment.csv not found in {OUT_DIR} or {DATA_DIR}")
        print("Place the 2022 base year Employment.csv in processed_data and re-run this cell.")
        df_employ = pd.DataFrame()

### 8f - Combined inputs summary

In [ ]:
school_cols = ["TAZ"] + [c for c in df_school.columns if c != "TAZ"]

df_summary = (
    df_socio
    .merge(df_overnight,           on="TAZ", how="outer")
    .merge(df_visitor_occ,         on="TAZ", how="outer")
    .merge(df_school[school_cols], on="TAZ", how="outer")
)
if len(df_employ):
    df_summary = df_summary.merge(df_employ, on="TAZ", how="outer")

df_summary = df_summary.fillna(0).sort_values("TAZ").reset_index(drop=True)
df_summary.to_csv(OUT_DIR / "inputs_summarized.csv", index=False)
print(f"inputs_summarized.csv -- {len(df_summary)} TAZs, {len(df_summary.columns)} columns")
df_summary.head()

---
## Stage 9 - QA & Basin Summary
Print basin-wide totals for sanity-checking against prior base years.

In [ ]:
print("=" * 55)
print(f"  BASE YEAR SUMMARY ({PARCEL_YEAR})")
print("=" * 55)
print(f"  Total parcels                : {len(sdfParcel):,}")
print(f"  Total residential units      : {sdfParcel['Residential_Units_Adjusted'].sum():,.0f}")
print(f"  Total TAU rooms              : {sdfParcel['TouristAccommodation_Units'].sum():,.0f}")
print(f"  VHR parcels                  : {(sdfParcel['VHR']=='Yes').sum():,}")
print(f"  Total occupied units         : {sdfParcel['OccupiedUnits'].sum():,.0f}")
print(f"  Total seasonal units         : {sdfParcel['SeasonalUnits'].sum():,.0f}")
print(f"  Total residents (people)     : {sdfParcel['People'].sum():,.0f}")
print(f"  Avg TAU occupancy rate       : {sdfParcel.loc[sdfParcel['TouristAccommodation_Units'].fillna(0)>0,'TAU_Occupancy_Rate'].mean():.4f}")
print(f"  Avg VHR occupancy rate       : {sdfParcel.loc[sdfParcel['VHR']=='Yes','VHR_Occupancy_Rate'].mean():.4f}")
print(f"  Avg household size (adjusted): {sdfParcel['PersonsPerUnit'].mean():.4f}")
print("=" * 55)

In [ ]:
# Check for parcels with no TAZ assignment
missing_taz = sdfParcel["TAZ"].isna().sum()
if missing_taz:
    print(f"WARNING: {missing_taz} parcels have no TAZ assignment")
    print(sdfParcel[sdfParcel["TAZ"].isna()][["APN","JURISDICTION","Residential_Units"]].head(20))
else:
    print("All parcels have TAZ assignments.")

# Confirm all output files exist
output_files = [
    "OvernightVisitorZonalData_Summer.csv",
    "VisitorOccupancyRates_Summer.csv",
    "SocioEcon_Summer.csv",
    "Employment.csv",
    "SchoolEnrollment.csv",
    "inputs_summarized.csv",
]
print("\nOutput files:")
for f in output_files:
    p = OUT_DIR / f
    status = f"{p.stat().st_size:,} bytes" if p.exists() else "MISSING"
    print(f"  {f:<45} {status}")